# EPIC: Explanation of Pretrained Image Classification Networks via Prototypes

Na podstawie artykułu: https://arxiv.org/pdf/2505.12897

---

<div style="text-align:center">

<img src="images/epic-example.png" width="750">

<p style="font-size:14px">
Źródło: <a href="https://arxiv.org/pdf/2505.12897">
EPIC: Explanation of Pretrained Image Classification Networks via Prototypes
</a>
</p>

</div>

## 1. Podstawy Matematyczne: Wybór Prototypów z Kanałów Map Cecha

Dla wejściowego obrazka $I$, ekstraktor cech modelu $\Phi_\Theta$ tworzy reprezentację w przestrzeni cech:
$$ Z_I = \Phi_\Theta(I) \in \mathbb{R}^{H \times W \times D} $$
Zamiast patrzeć na całą przestrzeń naraz, EPIC szuka obrazów treningowych, które aktywują konkretny **pojedynczy kanał** $k \in \{1, 2, \dots, D\}$.

### Aktywacja Kanału (Channel Activation)
Całkowita aktywacja kanału $k$ na całej mapie przestrzennej obrazu wynosi:
$$ activ(Z; k) = \sum_{h=1}^H \sum_{w=1}^W Z[h, w, k] $$
Wektor $Z[h, w] \in \mathbb{R}^D$ możemy traktować jako "piksel w przestrzeni cech".

### Wybór Prototypów (Top-m)
Szukamy $m$ (zazwyczaj $m=5$) obrazów ze zbioru treningowego, które najbardziej aktywują ten konkretny kanał $k$. Tak wyznaczone obrazy nazywamy **pozytywnymi prototypami** kanału $k$:
$$ Prot(k)_{pos} = \arg \text{top-m}_{I \in TrainSet} \, activ(Z_I ; k) $$

Analogicznie można definiować **prototypy negatywne** szukając obrazów o najmniejszych (najbardziej ujemnych) aktywacjach: $Prot(k)_{neg} = \arg \text{top-m}_{I \in TrainSet} -activ(Z_I ; k)$.

## 2. Czystość Prototypu (Purity of Prototype)

Klasyczna optymalizacja modeli skupia się tylko na wyniku predykcji. Prowadzi to do tzw. poplątanej reprezentacji (entangled representation), gdzie jedna logiczna cecha (np. "koło") jest rozproszona po wielu różnych kanałach sieci.

Idealny, "czysty" prototyp powinien aktywować się **tylko w jednym, konkretnym kanale**. EPIC wprowadza miarę czystości (*purity*), by to ocenić.

Dla wybranego kanału $k$ obrazu $I$, najpierw znajdujemy współrzędne $(h, w)$ piksela w przestrzeni cech o **największej aktywacji** dla tego kanału:
$$ \mathbb{N}^2 \ni (h, w) = \arg \max_{x,y} Z_I[x, y, k] $$

Tworzymy wektor tego "prototypowego piksela": $p = Z_I[h, w] \in \mathbb{R}^D$. 
Czystość definiujemy jako udział aktywacji kanału $k$ względem normy całego wektora:
$$ purity(I, k) = \frac{p_k}{\|p\|} \in [0, 1] $$

Jeśli $purity = 1$, prototyp jest **całkowicie czysty** – co oznacza, że poza kanałem $k$, wszystkie pozostałe kanały w tym wektorze mają wartość zero.

## 3. Moduł Rozplątujący i Niezmienniczość Predykcji (Disentanglement Module)

Aby wyjaśnienia były precyzyjne, EPIC zmusza przestrzeń cech do "rozplątania się" (maksymalizacji czystości prototypów). Jednakże ingerencja w cechy pre-trenowanego modelu mogłaby zepsuć jego skuteczność. Autorzy rozwiązują to w genialny sposób, wprowadzając uczoną, **odwracalną macierz** $U \in \mathbb{R}^{D \times D}$.

EPIC jest "wstrzykiwany" do modelu tuż przed warstwą Global Average Pooling. Aplikujemy liniową transformację $U$ do każdego piksela mapy cech (co oznaczamy operatorem $\circledast$). Aby oryginalna predykcja pozostała nienaruszona, końcowe wagi klasyfikatora (macierz $A$) mnożymy przez odwrotność naszej macierzy $U^{-1}$.

Oto kompletny przepływ (forward pass) zmodyfikowanej sieci:
1. Ekstrakcja z oryginalnej sieci: $$ Z = \Phi_\Theta(I) \in \mathbb{R}^{H \times W \times D} $$
2. Transformacja (Rozplątywanie): $$ \hat{Z} = U \circledast Z \in \mathbb{R}^{H \times W \times D}, \quad U \in \mathbb{R}^{D \times D} $$
3. Average Pooling: $$ v = \text{avg\_pool\_over\_channels}(\hat{Z}) \in \mathbb{R}^D $$
4. Nowy klasyfikator ($A' = AU^{-1}$): $$ w = A'v = (A U^{-1})v $$
5. Predykcja: $$ pred = \text{softmax}(w) $$

---

<div style="text-align:center">

<img src="images/epic-overview.png" width="750">

<p style="font-size:14px">
Źródło: <a href="https://arxiv.org/pdf/2505.12897">
EPIC: Explanation of Pretrained Image Classification Networks via Prototypes
</a>
</p>

</div>


---
### Twierdzenie (Remark 3.1) - Dowód Niezmienniczości
Dlaczego modyfikacja $U \circledast Z$ i wymnożenie klasyfikatora przez $U^{-1}$ zwraca dokładnie ten sam wynik, co przed modyfikacją? Wynika to bezpośrednio z faktu, że mnożenie macierzy jest rozdzielne względem dodawania (jakie występuje w operacji Average Pooling).

**Dowód:**
$$ U^{-1} \text{avg\_pool\_over\_channels}(U \circledast Z) $$
Rozwijamy definicję Average Poolingu (średnia ze wszystkich pikseli $H \times W$):
$$ = U^{-1} \left( \frac{1}{HW} \sum_{x,y} U Z[x, y] \right) $$
Wyciągamy stałą macierz $U$ przed sumę:
$$ = U^{-1} U \left( \frac{1}{HW} \sum_{x,y} Z[x, y] \right) $$
Ponieważ $U^{-1} U = I$ (macierz jednostkowa):
$$ = \text{avg\_pool\_over\_channels}(Z) $$

Dzięki temu prostemu trikowi matematycznemu, EPIC może uczyć macierz $U$, aby wyodrębniać krystalicznie czyste prototypy, **nie ryzykując pogorszenia skuteczności (accuracy) oryginalnego klasyfikatora!**